# Syria Black Marble Extraction

This notebook is converted from the full Syria extraction script. It downloads Black Marble rasters with `blackmarblepy`, computes nighttime lights statistics for Syria administrative boundaries, adds gas-flaring mask variants, and optionally extracts statistics around border crossing buffers.

Run the setup cells first, then run the extraction loop cell when you are ready to download/process data.

## Imports

Load Python packages used for Black Marble retrieval, geospatial processing, raster I/O, and zonal statistics.

In [ ]:
import os
from pathlib import Path
import geopandas as gpd
import pandas as pd
import pyreadr
from shapely.geometry import box

import numpy as np
from rasterstats import zonal_stats
from getpass import getpass
from dotenv import dotenv_values
from blackmarble import BlackMarble, Product
from blackmarble import raster, extract
import rioxarray
from exactextract import exact_extract

## Paths and NASA Earthdata Token

Find the repository root, set data paths, and load or prompt for the Black Marble bearer token.

In [ ]:
# Get repo root directory. In notebooks, __file__ is usually not defined, so walk up from cwd.
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
BOUNDARIES_DIR = REPO_ROOT / "boundaries"

secrets_path = Path.home() / ".config" / "ethiopia-economic-monitor" / "secrets.env"
secrets = dotenv_values(secrets_path)
blackmarble_token = secrets.get("BLACKMARBLE_TOKEN", "").strip()

if not blackmarble_token:
    blackmarble_token = getpass("Enter BlackMarble token (input hidden): ").strip()
    secrets_path.parent.mkdir(parents=True, exist_ok=True)
    secrets_path.write_text(f"BLACKMARBLE_TOKEN={blackmarble_token}\n")
    os.chmod(secrets_path, 0o600)

bm = BlackMarble(token=blackmarble_token)

## Load Syria Boundaries

Load Syria administrative boundary shapefiles and initialize the Black Marble client.

In [ ]:
adm0 = gpd.read_file(DATA_DIR / 'boundaries/syr_admin0.shp')
adm1 = gpd.read_file(DATA_DIR / 'boundaries/syr_admin1.shp')
adm2 = gpd.read_file(DATA_DIR / 'boundaries/syr_admin2.shp')
adm3 = gpd.read_file(DATA_DIR / 'boundaries/syr_admin3.shp')

## Border Crossing Buffers

Optionally create 1 km through 5 km buffers around Syria border crossing points.

In [ ]:
# ---- Border crossing point extraction toggle ----
# Set to True to extract NTL statistics around border crossing locations
# using 1 km, 2 km, 3 km, 4 km, and 5 km buffers (mean, sum, std of radiance)
EXTRACT_BORDER_CROSSINGS = True

if EXTRACT_BORDER_CROSSINGS:
    border_df = pd.read_excel(DATA_DIR / 'Syria Border Crossings.xlsx')
    border_gdf_base = gpd.GeoDataFrame(
        border_df,
        geometry=gpd.points_from_xy(border_df.longitude, border_df.latitude),
        crs='EPSG:4326'
    )
    # EPSG:32637 = UTM Zone 37N, appropriate metric CRS for Syria
    BUFFER_KM = [1, 2, 3, 4, 5]
    border_buffers = {}
    for _km in BUFFER_KM:
        _buf = border_gdf_base.copy()
        _buf['geometry'] = _buf.to_crs(epsg=32637).buffer(_km * 1000).to_crs('EPSG:4326')
        border_buffers[_km] = _buf

## Gas Flaring Buffers

Load gas flaring locations from RDS, create 5 km and 10 km buffers, and dissolve them into union geometries.

In [ ]:
gas_flaring_rds = pyreadr.read_r(str(DATA_DIR / 'ntl/gas_flare_locations.Rds'))
gas_flaring = gas_flaring_rds[None]  # Extract the dataframe from the RDS object
gas_flaring_10km = gpd.GeoDataFrame(gas_flaring, geometry=gpd.points_from_xy(gas_flaring.longitude, gas_flaring.latitude), crs='EPSG:4326')
gas_flaring_10km['geometry'] = gas_flaring_10km.to_crs(epsg=32647).buffer(10000).to_crs(gas_flaring_10km.crs)

gas_flaring_5km = gpd.GeoDataFrame(gas_flaring, geometry=gpd.points_from_xy(gas_flaring.longitude, gas_flaring.latitude), crs='EPSG:4326')
gas_flaring_5km['geometry'] = gas_flaring_5km.to_crs(epsg=32647).buffer(5000).to_crs(gas_flaring_5km.crs)

# Dissolve gas flaring buffers into single union geometries
gf_5km_union = gas_flaring_5km.union_all()
gf_10km_union = gas_flaring_10km.union_all()

# Ensure unions are in EPSG:4326
if hasattr(gf_5km_union, '__geo_interface__'):
    gf_5km_union = gpd.GeoSeries([gf_5km_union], crs=gas_flaring_5km.crs).to_crs('EPSG:4326')[0]
    gf_10km_union = gpd.GeoSeries([gf_10km_union], crs=gas_flaring_10km.crs).to_crs('EPSG:4326')[0]

## Helper Functions

Create geometry variants inside and outside a mask, used for gas-flaring-inclusive and gas-flaring-excluded extraction.

In [ ]:
def make_mask_variants(gdf, mask_union):
    """Create masked and inverse-masked versions of admin boundaries."""
    gdf = gdf.copy()
    gdf['_merge_id'] = range(len(gdf))

    gdf_in = gdf.copy()
    gdf_in['geometry'] = gdf_in.geometry.intersection(mask_union)
    gdf_in['geometry'] = gdf_in.geometry.make_valid()
    gdf_in = gdf_in[~gdf_in.is_empty & gdf_in.geometry.is_valid].reset_index(drop=True)

    gdf_out = gdf.copy()
    gdf_out['geometry'] = gdf_out.geometry.difference(mask_union)
    gdf_out['geometry'] = gdf_out.geometry.make_valid()
    gdf_out = gdf_out[~gdf_out.is_empty & gdf_out.geometry.is_valid].reset_index(drop=True)

    return gdf, gdf_in, gdf_out

## Prepare Boundaries and Date Ranges

Validate geometries, standardize CRS, and define the extraction date ranges.

In [ ]:
# Prepare admin boundaries
for gdf in [adm0, adm1, adm2, adm3]:
    if 'date' in gdf.columns:
        gdf.drop(columns='date', inplace=True)
    # Make sure geometries are valid
    gdf['geometry'] = gdf.geometry.make_valid()
    # Ensure proper CRS
    if gdf.crs is None:
        gdf.set_crs('EPSG:4326', inplace=True)
    elif gdf.crs.to_string() != 'EPSG:4326':
        gdf.to_crs('EPSG:4326', inplace=True)

start_date = "2023-01-01"
end_date_monthly = "2026-04-15"

end_date_annual = "2025-01-01"

## Download Rasters and Extract Statistics

Download missing Black Marble rasters, aggregate nighttime lights to admin boundaries, write CSV outputs, and extract border-crossing buffer statistics.

In [ ]:
for products in [Product.VNP46A2]:

    if products == Product.VNP46A4:
        print("Extracting VNP46A4 (annual composites)...")
        freq = "YS"
        folder = 'annual'
        end_date = end_date_annual

    elif products == Product.VNP46A3:
        print("Extracting VNP46A3 (monthly composites)...")
        freq = "MS"
        folder = 'monthly'
        end_date = end_date_monthly

    elif products == Product.VNP46A2:
        print("Extracting VNP46A2 (daily composites)...")
        freq = "D"
        folder = 'daily'
        end_date = end_date_monthly

    # Raw h5 files directory (existing files)
    raw_dir = DATA_DIR / f'ntl/raw/{folder}'
    raw_dir.mkdir(parents=True, exist_ok=True)

    # Processed rasters directory
    raster_dir = DATA_DIR / f'ntl/raw/'
    raster_dir.mkdir(parents=True, exist_ok=True)

    # Aggregated CSV output directory
    aggregated_dir = DATA_DIR / f'ntl/aggregated/{folder}'
    aggregated_dir.mkdir(parents=True, exist_ok=True)

    # Check if all outputs are up-to-date by verifying every expected date is present in each CSV
    def _csv_missing_dates(path, expected_dates):
        if not path.exists():
            return set(expected_dates)
        try:
            existing = pd.read_csv(path, usecols=['date'])
            print(existing['date'])
            existing_dates = set(pd.to_datetime(existing['date']))
        except Exception:
            return set(expected_dates)
        return set(expected_dates) - existing_dates

    full_date_range = pd.date_range(start_date, end_date, freq=freq)
    print(full_date_range)
    all_missing = [
        _csv_missing_dates(aggregated_dir / f'ntl_syr_adm{level}_{folder}.csv', full_date_range)
        for level in [0, 1, 2]
    ]
    print(all_missing)
    if all(len(missing) == 0 for missing in all_missing):
        print(f"All outputs up to date for {folder} - skipping {products.name}")
        continue

    # Check if processed rasters exist
    date_range = pd.date_range(start_date, end_date, freq=freq)
    product_name = products.name  # VNP46A3 or VNP46A4
    raster_files_exist = all(
        (raster_dir / f'{product_name}_{date.strftime("%Y%m%d")}.tif').exists()
        for date in date_range
    )


    if raster_files_exist:
        print(f"Rasters already exist for {folder} - skipping raster generation")
    else:
        print(f"Generating rasters for {folder}...")

        from concurrent.futures import ThreadPoolExecutor, as_completed

        def _save_rasters(rasters_ds):
            """Save all time slices from an xarray dataset as individual .tif files."""
            for var in rasters_ds.data_vars:
                da = rasters_ds[var]
                time_dim = 'time' if 'time' in da.dims else da.dims[0]
                for t_idx in range(da.sizes[time_dim]):
                    slice_da = da.isel({time_dim: t_idx})
                    date_val = pd.Timestamp(da[time_dim].values[t_idx])
                    date_str = date_val.strftime('%Y%m%d')
                    out_file = raster_dir / f'{product_name}_{date_str}.tif'
                    slice_da.rio.to_raster(out_file)

        adm0_bbox = gpd.GeoDataFrame(
            geometry=[box(*adm0.total_bounds)],
            crs=adm0.crs
        )

        def download_and_save(date, products, product_name):
            date_str = date.strftime('%Y%m%d')
            out_file = raster_dir / f'{product_name}_{date_str}.tif'
            if out_file.exists():
                return (date_str, 'exists')
            try:
                r = raster.bm_raster(
                    adm0_bbox,
                    products,
                    pd.date_range(date, date, freq=freq),
                    token=blackmarble_token,
                    output_directory=str(raw_dir),
                    output_skip_if_exists=True,
                    variable="Gap_Filled_DNB_BRDF-Corrected_NTL"
                )
                _save_rasters(r)
                return (date_str, 'downloaded')
            except ValueError:
                return (date_str, 'missing')
            except Exception as err:
                print(f"  Warning: unexpected error for {date_str}: {err}")
                return (date_str, 'error')

        # Parallel download using ThreadPoolExecutor
        max_workers = min(8, os.cpu_count() or 4)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(download_and_save, d, products, product_name): d for d in date_range}
            for future in as_completed(futures):
                date_str, status = future.result()
                if status == 'downloaded':
                    print(f"  Downloaded and saved raster for {date_str}")
                elif status == 'missing':
                    print(f"  Raster missing for {date_str} (NASA manifest)")
                elif status == 'exists':
                    pass  # already on disk
                elif status == 'error':
                    print(f"  Error downloading raster for {date_str}")

    # Build set of dates in the expected range for fast membership checks (used in fallback)
    adm_date_range = pd.date_range(start_date, end_date, freq=freq)
    adm_date_set = set(adm_date_range)

    for admin_level, eth_gdf in zip(
        [],
        [adm0]
    ):
        out_path = aggregated_dir / f'ntl_syr_adm{admin_level}_{folder}.csv'
        full_date_range = pd.date_range(start_date, end_date, freq=freq)

        # Determine which dates are missing from the existing CSV
        if out_path.exists():
            existing = pd.read_csv(out_path, usecols=['date'], dtype=str)
            # Convert to datetime, coerce errors to NaT, and only keep valid dates
            valid_dates = pd.to_datetime(existing['date'], errors='coerce')
            n_invalid = valid_dates.isna().sum()
            if n_invalid > 0:
                print(f"  Warning: {n_invalid} non-date values found in 'date' column of {out_path}. These will be ignored.")
            existing_dates = set(valid_dates[~valid_dates.isna()])
            missing_dates = [d for d in full_date_range if pd.Timestamp(d) not in existing_dates]
            if not missing_dates:
                print(f"  Admin level {admin_level}: all dates present — skipping.")
                continue
            date_range = pd.DatetimeIndex(missing_dates)
            print(f"  Admin level {admin_level}: {len(missing_dates)} dates missing — extracting those.")
        else:
            date_range = full_date_range
            print(f"Extracting for admin level {admin_level}...")

        extract_kwargs = dict(
            product_id=products,
            date_range=date_range,
            token=blackmarble_token,
            output_directory=str(raw_dir),
            output_skip_if_exists=True,
        )

        # Create masked variants for gas flaring
        gdf_with_id, gdf_gf_5km, gdf_nogf_5km = make_mask_variants(eth_gdf, gf_5km_union)
        _, gdf_gf_10km, gdf_nogf_10km = make_mask_variants(eth_gdf, gf_10km_union)

        # --- Primary: use bm_extract ---
        try:
            extracted = extract.bm_extract(gdf_with_id, **extract_kwargs)

            # Extract gas flaring masked variants
            for gdf_variant, prefix in [
                (gdf_gf_5km, 'ntl_gf_5km'),
                (gdf_nogf_5km, 'ntl_nogf_5km'),
                (gdf_gf_10km, 'ntl_gf_10km'),
                (gdf_nogf_10km, 'ntl_nogf_10km'),
            ]:
                col_name = f'{prefix}_sum'
                if not gdf_variant.empty:
                    try:
                        variant_extracted = extract.bm_extract(gdf_variant, **extract_kwargs)
                        variant_extracted = variant_extracted.rename(columns={'ntl_sum': col_name})
                        extracted = extracted.merge(
                            variant_extracted[['_merge_id', 'date', col_name]],
                            on=['_merge_id', 'date'], how='left'
                        )
                    except Exception as e:
                        print(f"  Warning: skipping {prefix} — {e}")
                        extracted[col_name] = 0.0
                else:
                    extracted[col_name] = 0.0

            for col in ['ntl_gf_5km_sum', 'ntl_nogf_5km_sum', 'ntl_gf_10km_sum', 'ntl_nogf_10km_sum']:
                extracted[col] = extracted[col].fillna(0.0)

            new_rows = extracted.drop(columns=['geometry', '_merge_id', '_join_id'], errors='ignore')
            # Robust CSV writing: align columns and concatenate
            if out_path.exists():
                existing_df = pd.read_csv(out_path)
                combined = pd.concat([existing_df, new_rows], ignore_index=True, sort=False)
                # Ensure consistent column order: put date first if present, then others sorted
                cols = ['date'] + [c for c in combined.columns if c != 'date'] if 'date' in combined.columns else list(combined.columns)
                combined = combined[cols]
                combined.to_csv(out_path, index=False)
            else:
                new_rows.to_csv(out_path, index=False)
            print(f"  Saved {len(new_rows)} new records to {out_path}")

        except ValueError as e:
            # --- Fallback: read .tif files already on disk, compute zonal stats directly ---
            print(f"  bm_extract failed ({e}). Falling back to on-disk rasters...")

            def _zonal_sum_by_id(gdf_v, arr, transform):
                if gdf_v.empty:
                    return {}
                stats = zonal_stats(gdf_v, arr, affine=transform, stats=['sum'],
                                    nodata=np.nan, all_touched=True)
                return {
                    int(gdf_v.iloc[i]['_merge_id']): (s['sum'] if s['sum'] is not None else 0.0)
                    for i, s in enumerate(stats)
                }

            missing_date_set = set(date_range)  # only process the missing dates
            all_records = []
            for tif_path in sorted(raster_dir.glob(f'{product_name}_*.tif')):
                try:
                    date_val = pd.Timestamp(tif_path.stem.split('_', 1)[1])
                except Exception:
                    continue
                if date_val not in missing_date_set:
                    continue
                try:
                    _da = rioxarray.open_rasterio(tif_path, masked=True).squeeze()
                    _arr = _da.values.astype('float64')
                    _transform = _da.rio.transform()
                except Exception as err:
                    print(f"  Warning: could not read {tif_path.name}: {err}")
                    continue

                total_stats = zonal_stats(gdf_with_id, _arr, affine=_transform,
                                          stats=['sum'], nodata=np.nan, all_touched=True)
                gf_5km_d    = _zonal_sum_by_id(gdf_gf_5km, _arr, _transform)
                nogf_5km_d  = _zonal_sum_by_id(gdf_nogf_5km, _arr, _transform)
                gf_10km_d   = _zonal_sum_by_id(gdf_gf_10km, _arr, _transform)
                nogf_10km_d = _zonal_sum_by_id(gdf_nogf_10km, _arr, _transform)

                for i, s in enumerate(total_stats):
                    mid = int(gdf_with_id.iloc[i]['_merge_id'])
                    all_records.append({
                        '_merge_id': mid,
                        'date': date_val,
                        'ntl_sum': s['sum'] if s['sum'] is not None else 0.0,
                        'ntl_gf_5km_sum': gf_5km_d.get(mid, 0.0),
                        'ntl_nogf_5km_sum': nogf_5km_d.get(mid, 0.0),
                        'ntl_gf_10km_sum': gf_10km_d.get(mid, 0.0),
                        'ntl_nogf_10km_sum': nogf_10km_d.get(mid, 0.0),
                    })

            if not all_records:
                print(f"  No raster files found for admin level {admin_level} — skipping.")
                continue

            fb_extracted = pd.DataFrame(all_records)
            attr_cols = [c for c in gdf_with_id.columns if c not in ['geometry', 'date']]
            fb_extracted = fb_extracted.merge(gdf_with_id[attr_cols], on='_merge_id', how='left')
            new_fb_rows = fb_extracted.drop(columns=['_merge_id', 'geometry', '_join_id'], errors='ignore')
            # Robust CSV writing: align columns and concatenate
            if out_path.exists():
                existing_df = pd.read_csv(out_path)
                combined = pd.concat([existing_df, new_fb_rows], ignore_index=True, sort=False)
                cols = ['date'] + [c for c in combined.columns if c != 'date'] if 'date' in combined.columns else list(combined.columns)
                combined = combined[cols]
                combined.to_csv(out_path, index=False)
            else:
                new_fb_rows.to_csv(out_path, index=False)
            print(f"  Saved {len(new_fb_rows)} new records to {out_path}")

    # ---- Extract NTL for border crossing buffers ----
    if EXTRACT_BORDER_CROSSINGS:
        bc_out_path = aggregated_dir / f'ntl_syr_border_crossings_{folder}.csv'
        full_bc_range = pd.date_range(start_date, end_date, freq=freq)
        if bc_out_path.exists():
            bc_existing = pd.read_csv(bc_out_path, usecols=['date'])
            bc_existing_dates = set(pd.to_datetime(bc_existing['date']))
            bc_missing = [d for d in full_bc_range if pd.Timestamp(d) not in bc_existing_dates]
            if not bc_missing:
                print(f"Skipping border crossings - all dates present.")
            else:
                print(f"Border crossings: {len(bc_missing)} dates missing — extracting those.")
            bc_date_range = pd.DatetimeIndex(bc_missing)
        else:
            bc_date_range = full_bc_range
            bc_missing = list(full_bc_range)  # non-empty so the block runs

        if bc_missing:
            print("Extracting NTL for border crossing buffers...")
            all_bc_records = []

            for _date in bc_date_range:
                _date_str = _date.strftime('%Y%m%d')
                _tif_path = raster_dir / f'{product_name}_{_date_str}.tif'
                if not _tif_path.exists():
                    continue

                for _km, _buf_gdf in border_buffers.items():
                    # Use exactextract for fractional pixel weighting — important for small buffers
                    _result = exact_extract(
                        str(_tif_path),
                        _buf_gdf,
                        ops=['sum', 'mean', 'stdev'],
                        output='pandas'
                    )

                    for _i, _row in _result.iterrows():
                        _crossing = border_df.iloc[_i]
                        all_bc_records.append({
                            'uid': _crossing['uid'],
                            'crossing_name': _crossing['crossing_name_1'],
                            'latitude': _crossing['latitude'],
                            'longitude': _crossing['longitude'],
                            'border_country': _crossing['border_country'],
                            'date': _date,
                            'buffer_km': _km,
                            'ntl_sum': _row.get('sum'),
                            'ntl_mean': _row.get('mean'),
                            'ntl_std': _row.get('stdev'),
                        })

            if all_bc_records:
                bc_df_out = pd.DataFrame(all_bc_records)
                # Robust CSV writing: align columns and concatenate
                if bc_out_path.exists():
                    existing_df = pd.read_csv(bc_out_path)
                    combined = pd.concat([existing_df, bc_df_out], ignore_index=True, sort=False)
                    cols = ['date'] + [c for c in combined.columns if c != 'date'] if 'date' in combined.columns else list(combined.columns)
                    combined = combined[cols]
                    combined.to_csv(bc_out_path, index=False)
                else:
                    bc_df_out.to_csv(bc_out_path, index=False)
                print(f"Saved {len(bc_df_out)} new border crossing records to {bc_out_path}")